# Stable Diffusion 1.5 Ultimate Workstation
This notebook has been upgraded to a full-featured AI Art Station:
- **Modes**: Text-to-Image & Image-to-Image.
- **Universal Loader**: CivitAI & Hugging Face support.
- **Prompt Manager**: Save and recall your favorite prompt combinations.
- **History Browser**: View and delete generated images directly in the UI.
- **Drive Integration**: Seamlessly save to Google Drive.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors gradio omegaconf invisible-watermark

In [ ]:
import torch
import gradio as gr
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, DPMSolverMultistepScheduler
import os
import requests
from datetime import datetime
from google.colab import drive
from PIL import Image
import json
import random
import shutil

# --- Global State ---
pipe_t2i = None
pipe_i2i = None
current_model_path = ""
drive_mounted = False
HISTORY_DIR = "/content/sd_history"
DRIVE_ROOT = "/content/drive/MyDrive"

if not os.path.exists(HISTORY_DIR):
    os.makedirs(HISTORY_DIR)

# --- Helper Classes ---
class PromptManager:
    def __init__(self):
        # Determine path dynamically based on Drive availability
        self.local_path = "/content/saved_prompts.json"
        self.drive_path = os.path.join(DRIVE_ROOT, "SD_Outputs", "saved_prompts.json")

    def get_filepath(self):
        # Prefer Drive if mounted
        if os.path.exists(DRIVE_ROOT):
             # Ensure directory exists
            drive_dir = os.path.dirname(self.drive_path)
            if not os.path.exists(drive_dir):
                try: os.makedirs(drive_dir)
                except: pass
            
            # Sync logic: If local exists but Drive doesn't, copy to Drive
            if os.path.exists(self.local_path) and not os.path.exists(self.drive_path):
                try: shutil.copy(self.local_path, self.drive_path)
                except: pass
            
            return self.drive_path
        return self.local_path

    def load(self):
        filepath = self.get_filepath()
        if not os.path.exists(filepath):
            return {}
        try:
            with open(filepath, 'r') as f:
                return json.load(f)
        except:
            return {}

    def save(self, name, prompt, neg_prompt):
        data = self.load()
        data[name] = {"prompt": prompt, "neg_prompt": neg_prompt}
        
        filepath = self.get_filepath()
        with open(filepath, 'w') as f:
            json.dump(data, f)
        
        # Return a Gradio update object to refresh the dropdown choices
        return gr.update(choices=list(data.keys()))

    def get_prompts_list(self):
        return list(self.load().keys())

    def get_prompt(self, name):
        data = self.load()
        if name in data:
            return data[name]["prompt"], data[name]["neg_prompt"]
        return "", ""

prompt_manager = PromptManager()

# --- Core Functions ---
def mount_google_drive():
    global drive_mounted
    if not drive_mounted:
        try:
            drive.mount('/content/drive')
            drive_mounted = True
            return True
        except Exception as e:
            print(f"Error mounting drive: {e}")
            return False
    return True

def load_model(model_path_or_url):
    global pipe_t2i, pipe_i2i, current_model_path
    
    if pipe_t2i is not None and model_path_or_url == current_model_path:
        return "Model already loaded."

    print(f"Loading model: {model_path_or_url}...")
    
    try:
        # Download/Path Resolution Logic
        final_path = model_path_or_url
        is_repo = not model_path_or_url.startswith("http") and "/" in model_path_or_url
        
        if not is_repo:
            if model_path_or_url.startswith("http"):
                filename = "custom_model.safetensors"
                print("Downloading model...")
                # Handle CivitAI or generic URLs
                response = requests.get(model_path_or_url, stream=True)
                response.raise_for_status()
                if "content-disposition" in response.headers:
                    import re
                    fname = re.findall("filename=(.+)", response.headers["content-disposition"])
                    if fname:
                        filename = fname[0].strip('"')
                
                with open(filename, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                final_path = filename

        # Load Pipeline (Text2Img)
        if is_repo:
            pipe_t2i = StableDiffusionPipeline.from_pretrained(
                final_path, torch_dtype=torch.float16, use_safetensors=True
            )
        else:
            pipe_t2i = StableDiffusionPipeline.from_single_file(
                final_path, torch_dtype=torch.float16
            )

        # Configure Scheduler
        pipe_t2i.scheduler = DPMSolverMultistepScheduler.from_config(
            pipe_t2i.scheduler.config, 
            use_karras_sigmas=True, 
            algorithm_type="dpmsolver++"
        )
        
        # Optimizations
        pipe_t2i.enable_model_cpu_offload()
        pipe_t2i.safety_checker = None
        
        # Create Img2Img Pipeline sharing components
        pipe_i2i = StableDiffusionImg2ImgPipeline(vae=pipe_t2i.vae, text_encoder=pipe_t2i.text_encoder, tokenizer=pipe_t2i.tokenizer, unet=pipe_t2i.unet, scheduler=pipe_t2i.scheduler, safety_checker=None, feature_extractor=None)
        
        current_model_path = model_path_or_url
        return "Model loaded successfully!"
        
    except Exception as e:
        return f"Error loading model: {str(e)}"

def generate(mode, prompt, neg_prompt, img_input, steps, cfg, width, height, seed, randomize, num_images, save_drive, drive_folder, strength=0.75):
    global pipe_t2i, pipe_i2i
    if pipe_t2i is None:
        return [None] * num_images, seed

    # Handle Seed
    if randomize:
        seed = random.randint(0, 2147483647)
    
    generator = torch.Generator(device="cpu").manual_seed(int(seed))
    
    print(f"Generating in mode: {mode}")
    
    # Generate
    if mode == "txt2img":
        images = pipe_t2i(
            prompt, negative_prompt=neg_prompt, num_inference_steps=steps, 
            guidance_scale=cfg, width=width, height=height, 
            num_images_per_prompt=num_images, generator=generator
        ).images
    else: # img2img
        if img_input is None:
            print("Error: No input image for img2img")
            return [], seed
        # Resize input image to match target dimensions to avoid mismatches
        init_image = img_input.resize((width, height))
        images = pipe_i2i(
            prompt, image=init_image, negative_prompt=neg_prompt, 
            num_inference_steps=steps, guidance_scale=cfg, strength=strength,
            num_images_per_prompt=num_images, generator=generator
        ).images

    # Save Logic
    saved_paths = []
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. Save to History (Colab Local)
    for i, img in enumerate(images):
        fname = f"sd_{timestamp}_{i}.png"
        img.save(os.path.join(HISTORY_DIR, fname))
    
    # 2. Save to Drive (Optional)
    if save_drive and mount_google_drive():
        drive_path = f"/content/drive/MyDrive/{drive_folder}"
        os.makedirs(drive_path, exist_ok=True)
        for i, img in enumerate(images):
            img.save(f"{drive_path}/sd_{timestamp}_{i}.png")

    return images, seed

def get_history_images():
    files = sorted(os.listdir(HISTORY_DIR), reverse=True)
    paths = [os.path.join(HISTORY_DIR, f) for f in files if f.endswith('.png')]
    return paths

def select_image(evt: gr.SelectData):
    # Callback to update the state with the selected image path
    return evt.value['image']['path']

def delete_image(path):
    if not path:
        return get_history_images(), "No image selected."
    
    try:   
        # Gradio might provide a temp path. We need to find the real file in HISTORY_DIR.
        filename = os.path.basename(path)
        real_path = os.path.join(HISTORY_DIR, filename)
        
        if os.path.exists(real_path):
            os.remove(real_path)
            return get_history_images(), "Image deleted."
        else:
            return get_history_images(), f"Image not found at {real_path}"
    except Exception as e:
        return get_history_images(), f"Error: {str(e)}"

# --- UI Layout ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Stable Diffusion 1.5 Ultimate Workstation")
    
    # Global Settings
    with gr.Row():
        model_input = gr.Textbox(label="Model URL/ID", value="runwayml/stable-diffusion-v1-5", scale=4)
        load_btn = gr.Button("Load Model", variant="secondary", scale=1)
    status_text = gr.Textbox(label="System Status", interactive=False)

    with gr.Tabs():
        # --- Tab 1: Generation ---
        with gr.Tab("Generate"):
            with gr.Row():
                with gr.Column(scale=3):
                    prompt = gr.Textbox(label="Prompt", lines=2, placeholder="Describe your image...")
                    neg_prompt = gr.Textbox(label="Negative Prompt", lines=2, value="low quality, bad anatomy, worst quality")
                with gr.Column(scale=1):
                    gen_btn = gr.Button("GENERATE", variant="primary", size="lg")
                    random_seed = gr.Checkbox(label="Randomize Seed", value=True)
                    seed = gr.Number(label="Seed", value=42, precision=0)

            # Mode Management
            mode_state = gr.State(value="txt2img")
            
            with gr.Tabs() as mode_tabs:
                with gr.Tab("Text-to-Image", id="t2i_tab"):
                    pass # No special inputs needed
                with gr.Tab("Image-to-Image", id="i2i_tab"):
                    img_input = gr.Image(label="Input Image", type="pil")
                    denoise = gr.Slider(label="Denoising Strength", minimum=0.0, maximum=1.0, value=0.75)

            with gr.Accordion("Advanced Settings", open=False):
                with gr.Row():
                    width = gr.Slider(label="Width", minimum=256, maximum=1024, step=64, value=512)
                    height = gr.Slider(label="Height", minimum=256, maximum=1024, step=64, value=512)
                    steps = gr.Slider(label="Steps", minimum=10, maximum=100, step=1, value=25)
                    cfg = gr.Slider(label="CFG Scale", minimum=1, maximum=20, step=0.5, value=7.5)
                    batch_size = gr.Slider(label="Batch Size", minimum=1, maximum=4, step=1, value=1)
            
            with gr.Row():
                save_drive = gr.Checkbox(label="Save to Drive", value=False)
                drive_folder = gr.Textbox(label="Drive Folder", value="SD_Outputs")
                
            gallery = gr.Gallery(label="Output", columns=2)

        # --- Tab 2: History ---
        with gr.Tab("History Browser"):
            with gr.Row():
                refresh_hist_btn = gr.Button("Refresh History")
                del_img_btn = gr.Button("Delete Selected Image", variant="stop")
            hist_gallery = gr.Gallery(label="Saved Images", columns=4, allow_preview=True)
            hist_msg = gr.Textbox(label="Message")
            selected_img_path = gr.State(value="")

        # --- Tab 3: Prompt Manager ---
        with gr.Tab("Saved Prompts"):
            style_name = gr.Textbox(label="Style Name")
            save_style_btn = gr.Button("Save Current Prompts")
            saved_styles_drop = gr.Dropdown(label="Load Style", choices=prompt_manager.get_prompts_list())
            load_style_btn = gr.Button("Load Selected")

    # --- Wiring ---
    load_btn.click(load_model, [model_input], [status_text])
    
    # Mode Switch Logic - Update state based on tab selection
    # We use a trick: Bind the tab selection to update the mode_state
    t2i_tab = mode_tabs.children[0]
    i2i_tab = mode_tabs.children[1]
    
    t2i_tab.select(fn=lambda: "txt2img", outputs=mode_state)
    i2i_tab.select(fn=lambda: "img2img", outputs=mode_state)
    
    # Generate Trigger
    gen_btn.click(
        fn=generate,
        inputs=[mode_state, prompt, neg_prompt, img_input, steps, cfg, width, height, seed, random_seed, batch_size, save_drive, drive_folder, denoise],
        outputs=[gallery, seed]
    )
    
    # History Logic
    refresh_hist_btn.click(get_history_images, outputs=[hist_gallery])
    
    # Select image -> Store path in state (DO NOT DELETE)
    hist_gallery.select(select_image, None, selected_img_path)
    
    # Click Delete -> Read path from state -> Delete -> Refresh
    del_img_btn.click(delete_image, inputs=[selected_img_path], outputs=[hist_gallery, hist_msg])

    # Prompt Manager Logic
    save_style_btn.click(
        fn=lambda n, p, np: prompt_manager.save(n, p, np),
        inputs=[style_name, prompt, neg_prompt],
        outputs=[saved_styles_drop]
    )
    
    def load_style_wrapper(name):
        p, np = prompt_manager.get_prompt(name)
        return p, np

    load_style_btn.click(
        fn=load_style_wrapper,
        inputs=[saved_styles_drop],
        outputs=[prompt, neg_prompt]
    )

demo.launch(share=True, debug=True)